# Defect formation energies

In [ ]:
from workflows.pyiron.build import bulk, repeat
from workflows.pyiron.evcurves import relax_structure
from workflows.pyiron.pointdefects import (
    create_interstitial,
    create_substitutional,
    calculate_vacancy_formation_energy,
    calculate_substitutional_formation_energy,
    calculate_interstitial_formation_energy,
)
from pyiron_workflow import Workflow
from conceptual_dictionary import ConceptualDict


In [ ]:
cd = ConceptualDict()

## Iron bulk

In [ ]:
#need grace installed
pair_style  = "grace"
pair_coeff  = "* * GRACE-2L-OMAT Fe"

In [ ]:
wf = Workflow('pFe')
wf.bulk = bulk('Fe', cubic=True, cdict=cd)
wf.structure = repeat(wf.bulk, repetitions=(5,5,5), cdict=cd)
wf.relax = relax_structure(wf.structure, 
                            pair_style, pair_coeff, cdict=cd)
rbulkFe = wf.run()

## Silicon bulk

In [ ]:
pair_style  = "grace"
pair_coeff  = "* * GRACE-2L-OMAT Si"

In [ ]:
wf = Workflow('pSi')
wf.bulk = bulk('Si', cubic=True, cdict=cd)
wf.structure = repeat(wf.bulk, repetitions=(5,5,5), cdict=cd)
wf.relax = relax_structure(wf.structure, 
                            pair_style, pair_coeff, cdict=cd)
rbulkSi = wf.run()

## Defect structure

In [ ]:
wf = Workflow('pFeSi')
wf.bulk = bulk('Fe', cubic=True, cdict=cd)
wf.repeat_bulk = repeat(wf.bulk, repetitions=(5,5,5), cdict=cd)
wf.structure = create_substitutional(wf.repeat_bulk, 'Si', cdict=cd)
wf.relax = relax_structure(wf.structure, 
                            pair_style, pair_coeff, cdict=cd)
rdefect = wf.run()


In [ ]:
# Extract energies and structures from workflow results.
# relax_structure outputs: final_structure, ecoh (eV/atom), vol (Å³/atom)
bulk_Fe_struct = rbulkFe['relax__final_structure']
e_bulk_Fe      = rbulkFe['relax__ecoh']

bulk_Si_struct = rbulkSi['relax__final_structure']
e_bulk_Si      = rbulkSi['relax__ecoh']

defect_struct  = rdefect['relax__final_structure']
e_defect       = rdefect['relax__ecoh']

# Substitutional formation energy: Fe(N-1)Si(1) vs pure Fe bulk
# mu_host = e_coh(Fe), mu_impurity = e_coh(Si)
e_sub = calculate_substitutional_formation_energy(
    bulk_Fe_struct, defect_struct,
    e_bulk_Fe, e_defect,
    mu_host=e_bulk_Fe,
    mu_impurity=e_bulk_Si,
    cdict=cd,
)
print(f"Substitutional formation energy (Fe→Si): {e_sub:.4f} eV")


In [ ]:
cd.to_yaml('defect_formation.yaml')
